# 02_Clustering_scRNA - Clustering Analysis for scRNA-seq

This notebook performs clustering analysis on the scRNA-seq dataset:
- Preprocessing: log1p, HVG selection, scaling, PCA
- Clustering: KMeans, Agglomerative, DBSCAN
- Evaluation: Silhouette score, Davies-Bouldin, Adjusted Rand Index (if labels available)
- Visualizations: PCA/UMAP scatter colored by cluster


In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.cluster import KMeans, AgglomerativeClustering, DBSCAN
from sklearn.metrics import silhouette_score, davies_bouldin_score, adjusted_rand_score
import warnings
warnings.filterwarnings('ignore')

X = pd.read_csv('data/scRNA_data.csv', index_col=0)
try:
    y = pd.read_csv('data/sc_RNA_labels.csv', index_col=0)
except Exception:
    y = None

print('Loaded:', X.shape)
if y is not None:
    print('Labels loaded:', y.shape)


In [ ]:
# Preprocessing: log1p, HVG, scaling, PCA
X_log = np.log1p(X)

gene_vars = X_log.var(axis=0).sort_values(ascending=False)

n_hvg = 2000 if X_log.shape[1] >= 2000 else X_log.shape[1]
X_hvg = X_log[gene_vars.head(n_hvg).index]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X_hvg)

# PCA to 50 components for clustering (and 2 for plotting)
pca = PCA(n_components=50, random_state=42)
X_pca = pca.fit_transform(X_scaled)
pca2 = PCA(n_components=2, random_state=42)
X_pca2 = pca2.fit_transform(X_scaled)

print('Preprocessing done:', X_pca.shape)
print('Explained variance (first 10 PCs):', pca.explained_variance_ratio_[:10])

In [ ]:
# Automatic search for optimal k using silhouette score
results = []
for k in range(2, 21):
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = km.fit_predict(X_pca)
    sil = silhouette_score(X_pca, labels)
    db = davies_bouldin_score(X_pca, labels)
    results.append({'k': k, 'silhouette': sil, 'davies_bouldin': db})

import pandas as pd

df_k = pd.DataFrame(results).set_index('k')
print(df_k)

best_k = int(df_k['silhouette'].idxmax())
print(f'Best k by silhouette: {best_k}')

plt.figure(figsize=(8,4))
plt.plot(df_k.index, df_k['silhouette'], marker='o', label='Silhouette')
plt.xlabel('k')
plt.ylabel('Silhouette Score')
plt.title('Silhouette vs k (2-20)')
plt.grid(True)
plt.show()

## Interpretation & Next Steps
- Use the `df_k` table to choose the number of clusters objectively (silhouette high, DB low).
- Compare clusters against known labels (ARI) to see if clusters match biological classes.
- Visualize top marker genes per cluster by comparing mean expression per cluster (not included here).
- For publication-quality clustering, consider UMAP for visualization and Leiden/Leiden-like methods (requires scanpy/igraph).
- Save cluster assignments to disk for downstream analysis.


In [ ]:
# Fit best KMeans, evaluate and visualize (PCA + UMAP)
km_best = KMeans(n_clusters=best_k, random_state=42, n_init=50)
km_labels = km_best.fit_predict(X_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_pca2[:,0], y=X_pca2[:,1], hue=km_labels, palette='tab10', s=40)
plt.title(f'KMeans (k={best_k}) on PCA(2)')
plt.xlabel('PC1')
plt.ylabel('PC2')
plt.legend(title='cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

print('Silhouette (best KMeans):', silhouette_score(X_pca, km_labels))
print('Davies-Bouldin (best KMeans):', davies_bouldin_score(X_pca, km_labels))
if y is not None:
    true = y.iloc[:,0].values
    ari = adjusted_rand_score(true, km_labels)
    print('Adjusted Rand Index (KMeans vs true):', ari)

# Agglomerative Clustering metrics
agg = AgglomerativeClustering(n_clusters=best_k)
agg_labels = agg.fit_predict(X_pca)
print('Agglomerative silhouette:', silhouette_score(X_pca, agg_labels))
print('Agglomerative DB:', davies_bouldin_score(X_pca, agg_labels))

# DBSCAN: try a couple of eps values and min_samples
for eps in [0.5, 1.0, 1.5]:
    dbs = DBSCAN(eps=eps, min_samples=5)
    dbs_labels = dbs.fit_predict(X_pca)
    n_clusters = len(set(dbs_labels)) - (1 if -1 in dbs_labels else 0)
    if len(set(dbs_labels)) <= 1:
        print(f'DBSCAN eps={eps}: single cluster or noise only (labels unique={set(dbs_labels)})')
        continue
    print(f'DBSCAN eps={eps}: clusters={n_clusters}, silhouette={silhouette_score(X_pca, dbs_labels):.4f}, db={davies_bouldin_score(X_pca, dbs_labels):.4f}')

# UMAP visualization (requires umap-learn)
try:
    import umap
except Exception:
    raise ImportError('Please install umap-learn: pip install umap-learn')

reducer = umap.UMAP(n_components=2, random_state=42)
X_umap = reducer.fit_transform(X_pca)

plt.figure(figsize=(8,6))
sns.scatterplot(x=X_umap[:,0], y=X_umap[:,1], hue=km_labels, palette='tab10', s=40)
plt.title('KMeans clusters on UMAP(2)')
plt.xlabel('UMAP1')
plt.ylabel('UMAP2')
plt.legend(title='cluster', bbox_to_anchor=(1.05, 1), loc='upper left')
plt.tight_layout()
plt.show()

# Save cluster assignments
clusters_df = pd.DataFrame({'cluster_kmeans': km_labels}, index=X.index)
clusters_df.to_csv('data/cluster_assignments_kmeans.csv')
print('Saved cluster assignments to data/cluster_assignments_kmeans.csv')
